In [0]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier

from pyspark.sql import functions as F

In [0]:
SOURCE_TABLE = "health.gold_serving_risk_label_v2"      # bảng có nhãn thật risk_label_v2
OUT_TABLE    = "health.gold_serving_risk_label_xgb_v2"  # bảng output serving cho XGB

LABEL_COL = "risk_label_v2"

FEATURE_NUM = [
    "avg_hr","avg_sys","avg_dia","avg_spo2",
    "cholesterol","glucose","hemoglobin","age_est"
]
FEATURE_CAT = ["gender","region","state","country"]  # bạn có thể bỏ bớt nếu muốn

KEY_COLS = ["patient_id", "date"]


In [0]:
cols_needed = KEY_COLS + [LABEL_COL] + FEATURE_NUM + FEATURE_CAT

sdf = (
    spark.table(SOURCE_TABLE)
    .select(*cols_needed)
    .filter(F.col(LABEL_COL).isNotNull())
)

print("Rows:", sdf.count())
pdf = sdf.toPandas()

# xử lý type cho chắc
pdf["age_est"] = pdf["age_est"].astype("int64")  # spark decimal(13,0) -> int
# date đã là date; nếu bị object/string thì mở dòng dưới:
# pdf["date"] = pd.to_datetime(pdf["date"]).dt.date

In [0]:
X = pdf[FEATURE_NUM + FEATURE_CAT].copy()
y = pdf[LABEL_COL].astype(str).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [0]:
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURE_CAT),
        ("num", "passthrough", FEATURE_NUM),
    ]
)

xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=3,                 # Normal/Elevated/High
    n_estimators=300,
    max_depth=5,
    learning_rate=0.07,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=42,
    eval_metric="mlogloss"
)

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("xgb", xgb)
])

pipe.fit(X_train, y_train)

# quick eval
pred_test = pipe.predict(X_test)
acc = accuracy_score(y_test, pred_test)
print("accuracy:", round(acc, 4))
print("\nclassification report:\n", classification_report(y_test, pred_test))
print("confusion_matrix:\n", confusion_matrix(y_test, pred_test))

In [0]:
proba_all = pipe.predict_proba(X)                 # shape (N, 3)
pred_all  = pipe.predict(X)

# quan trọng: đảm bảo đúng thứ tự class
classes = list(pipe.named_steps["xgb"].classes_)  # ví dụ ['Elevated','High','Normal'] hoặc khác
print("Class order from model:", classes)

# build prob theo đúng tên cột Normal/Elevated/High
proba_df = pd.DataFrame(proba_all, columns=[f"prob_{c.lower()}" for c in classes])

# nếu classes không đúng thứ tự Normal/Elevated/High, ta vẫn map đúng theo tên:
def get_prob_col(cls_name: str):
    return f"prob_{cls_name.lower()}"

pdf_out = pdf[KEY_COLS + [LABEL_COL] + FEATURE_NUM + FEATURE_CAT].copy()
pdf_out["risk_label_pred"] = pred_all

# tạo 3 cột xác suất luôn đủ
pdf_out["prob_normal"]   = proba_df.get(get_prob_col("Normal"),   0.0)
pdf_out["prob_elevated"] = proba_df.get(get_prob_col("Elevated"), 0.0)
pdf_out["prob_high"]     = proba_df.get(get_prob_col("High"),     0.0)

# risk_score (ví dụ dạng expected class index: Normal=0, Elevated=1, High=2)
pdf_out["risk_score_v2"] = (
    0.0 * pdf_out["prob_normal"]
    + 1.0 * pdf_out["prob_elevated"]
    + 2.0 * pdf_out["prob_high"]
)